# Armado de archivo de datos
# temperaturas_unificadas.csv

Los datos originales consisten en una carpeta con 1.702 archivos.
Cada archivo corresponde a una noche de observación.
El nombre de cada archivo contiene la fecha de observación.
Cada fila contiene, el instante en el que se comenzó la observación, una serie de variables atmosféricas, geométricas e instrumentales asociadas a las observaciones y al procesamiento de los datos, la altura y la temperatura.
Cada perfil de temperatura, que sería una observación completa, consiste en 1600 filas que corresponden a las temperaturas medidas desde los 68 m hasta los 159.968 m. Dentro de estas filas, los datos relevantes que cambian son la altura y la temperatura.

Cada noche puede tener varios perfiles de temperatura.

El siguiente código arma un único archivo con todas las observaciones de la carpeta original.

La estructura del nuevo archivo será la siguiente:
El nombre del **archivo** original.
La **fecha** de la observación, que se obtiene del nombre del archivo original.
El **time** que identifica el instante en el que se comenzó la observación.
Las siguientes columnas corresponden a las alturas de cada medición, que son iguales en todos los perfiles, 1.600 en total, **68, 168, 268, ..., 159.768, 159.868, 159.968**. A cada una le corresponde la temperatura en K, grados Kelvin. El 0 implica que no hay medición, es un NaN.

La combinación de *fecha* y *time* hacen que cada perfil sea único.


A partir de la carpeta de CSVs armamos un único archivo con las siguientes columnas
| archivo               | fecha      |     time |  68 | 168 | 268 | 368 | ... | 159768 | 159868 | 159968 |
| --------------------- | ---------- | -------: | --: | --: | --: | --: | --: |    --: |    --: |    --: |
| 20171128-0056_T60Z900 | 2017-11-28 |  9601000 |   0 |   0 |   0 |   0 | ... |    ... |    ... |    ... |
| 20171128-0056_T60Z900 | 2017-11-28 | 10501000 | ... | ... | ... | ... | ... |    ... |    ... |    ... |
| 20171130-0051_T60Z900 | 2017-11-30 |      ... | ... | ... | ... | ... | ... |    ... |    ... |    ... | 


In [1]:
from pathlib import Path
import pandas as pd

# carpeta con los csv
carpeta = Path(r"C:\Users\orlan\OneDrive\Documents\Tecnicatura\2A1C - Aprendizaje Automatico\Estacion Astronomica RG - Lidar GW\data\data")

dfs = []

for archivo in carpeta.glob("*.csv"):

    print(f"Leyendo {archivo.name}")

    # leer csv
    df = pd.read_csv(archivo)

    # limpiar nombres de columnas
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
    )

    # verificar columnas
    columnas_requeridas = [
        "time",
        "altitude",
        "temperature"
    ]

    faltantes = [
        c for c in columnas_requeridas
        if c not in df.columns
    ]

    if faltantes:
        print(f"Faltan columnas en {archivo.name}: {faltantes}")
        print("Columnas encontradas:", list(df.columns))
        continue

    # conservar columnas
    df = df[
        ["time", "altitude", "temperature"]
    ]

    # pivot altitude -> columnas
    df_pivot = (
        df.pivot(
            index="time",
            columns="altitude",
            values="temperature"
        )
        .reset_index()
    )

    # nombre archivo
    nombre = archivo.stem

    # fecha YYYYMMDD
    fecha = pd.to_datetime(
        nombre[:8],
        format="%Y%m%d"
    ).date()

    # insertar columnas
    df_pivot.insert(0, "archivo", nombre)
    df_pivot.insert(1, "fecha", fecha)

    dfs.append(df_pivot)

# unir
resultado = pd.concat(
    dfs,
    ignore_index=True
)

# ordenar columnas
cols_fijas = ["archivo", "fecha", "time"]
cols_altura = sorted(
    [c for c in resultado.columns
     if c not in cols_fijas]
)

resultado = resultado[
    cols_fijas + cols_altura
]

# guardar
resultado.to_csv(
    "temperaturas_unificadas.csv",
    index=False
)

print("Proceso terminado.")

Leyendo 20171124-0102_T60Z900.csv
Leyendo 20171127-0052_T60Z900.csv
Leyendo 20171128-0056_T60Z900.csv
Leyendo 20171130-0051_T60Z900.csv
Leyendo 20171205-0559_T60Z900.csv
Leyendo 20171206-0557_T60Z900.csv
Leyendo 20171210-0128_T60Z900.csv
Leyendo 20171211-0537_T60Z900.csv
Leyendo 20171217-0230_T60Z900.csv
Leyendo 20171219-0340_T60Z900.csv
Leyendo 20171226-0225_T60Z900.csv
Leyendo 20171229-0217_T60Z900.csv
Leyendo 20171230-0244_T60Z900.csv
Leyendo 20180101-0247_T60Z900.csv
Leyendo 20180106-0206_T60Z900.csv
Leyendo 20180109-0348_T60Z900.csv
Leyendo 20180110-0414_T60Z900.csv
Leyendo 20180111-0502_T60Z900.csv
Leyendo 20180115-0124_T60Z900.csv
Leyendo 20180117-0130_T60Z900.csv
Leyendo 20180118-0253_T60Z900.csv
Leyendo 20180131-0130_T60Z900.csv
Leyendo 20180206-0420_T60Z900.csv
Leyendo 20180207-0234_T60Z900.csv
Leyendo 20180208-0045_T60Z900.csv
Leyendo 20180209-0122_T60Z900.csv
Leyendo 20180210-0115_T60Z900.csv
Leyendo 20180211-0145_T60Z900.csv
Leyendo 20180212-0248_T60Z900.csv
Leyendo 201802